# 03 — EDA: GRAPE Dataset

GRAPE is our **primary dataset** — the only open-access multimodal glaucoma progression dataset with annotated VF progression labels.

Explore: fundus images, VF series, clinical data, and progression annotations.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

GRAPE_DIR = Path("../data/raw/grape")
print("GRAPE dir exists:", GRAPE_DIR.exists())

if GRAPE_DIR.exists():
    for p in sorted(GRAPE_DIR.rglob("*"))[:20]:
        if p.is_file():
            print(f"  {p.relative_to(GRAPE_DIR)}")

## 1. Load Clinical & VF Data

In [ ]:
# Load GRAPE clinical/VF CSV files
csv_files = sorted(GRAPE_DIR.rglob("*.csv")) if GRAPE_DIR.exists() else []
print(f"CSV files: {len(csv_files)}")
for f in csv_files:
    print(f"  {f.name}")

In [ ]:
# Try loading patient-level clinical data
clinical_file = next((f for f in csv_files if "clinical" in f.name.lower() or "patient" in f.name.lower()), None)
vf_file = next((f for f in csv_files if "vf" in f.name.lower() or "visual" in f.name.lower()), None)

print(f"Clinical file: {clinical_file}")
print(f"VF file: {vf_file}")

# Load if found
clinical_df = pl.read_csv(clinical_file) if clinical_file else None
vf_df = pl.read_csv(vf_file) if vf_file else None

if clinical_df is not None:
    print(f"\nClinical data: {clinical_df.shape}")
    print(clinical_df.head(3))

if vf_df is not None:
    print(f"\nVF data: {vf_df.shape}")
    print(vf_df.head(3))

In [ ]:
# Fall back to any CSV for structure exploration
if csv_files:
    for csv in csv_files[:3]:
        df = pl.read_csv(csv)
        print(f"\n{csv.name}: shape={df.shape}")
        print("Columns:", df.columns[:15])
        print(df.head(2))

## 2. Progression Annotations

In [ ]:
# Look for progression labels (this is what makes GRAPE special)
all_dfs = {f.name: pl.read_csv(f) for f in csv_files}

for name, df in all_dfs.items():
    progression_cols = [c for c in df.columns if any(kw in c.lower() 
                         for kw in ["progress", "rate", "slope", "change", "dbloss"])]
    if progression_cols:
        print(f"\n[{name}] Progression columns: {progression_cols}")
        print(df[progression_cols].describe())

## 3. Fundus Image Exploration

In [ ]:
# Find fundus images
img_exts = {".jpg", ".jpeg", ".png", ".tiff", ".bmp"}
image_files = [p for p in GRAPE_DIR.rglob("*") if p.suffix.lower() in img_exts] if GRAPE_DIR.exists() else []
print(f"Images found: {len(image_files)}")

if image_files:
    # Sample 6 random images
    sample_imgs = np.random.choice(image_files, min(6, len(image_files)), replace=False)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, img_path in zip(axes.flat, sample_imgs):
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(img_path.stem[:20], fontsize=8)
        ax.axis("off")
    plt.suptitle("GRAPE Fundus Images (random sample)")
    plt.tight_layout()
    plt.savefig("../reports/eda_grape_fundus_sample.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No images found — they may need to be downloaded separately")
    print("Check the GRAPE README for image download instructions")

In [ ]:
# Image statistics if available
if image_files:
    shapes = []
    for img_path in image_files[:50]:  # sample 50
        try:
            img = mpimg.imread(str(img_path))
            shapes.append(img.shape)
        except Exception:
            pass
    
    unique_shapes = set(shapes)
    print(f"Image shapes (sample of {len(shapes)}): {unique_shapes}")
    heights = [s[0] for s in shapes]
    widths  = [s[1] for s in shapes]
    print(f"Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")
    print(f"Width:  min={min(widths)},  max={max(widths)},  mean={np.mean(widths):.0f}")

## 4. VF Time Series Visualization

In [ ]:
# Plot longitudinal MD trajectories for sample patients
if vf_df is not None:
    pid_col = next((c for c in vf_df.columns if "patient" in c.lower()), vf_df.columns[0])
    date_col = next((c for c in vf_df.columns if "date" in c.lower()), None)
    md_col = next((c for c in vf_df.columns if c.lower() == "md"), None)
    
    if date_col and md_col:
        # Get patients with ≥5 visits
        vc = vf_df.group_by(pid_col).len()
        good_patients = vc.filter(pl.col("len") >= 5)[pid_col].to_list()[:8]
        
        fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
        for ax, pid in zip(axes.flat, good_patients):
            pdata = vf_df.filter(pl.col(pid_col) == pid).sort(date_col)
            times = np.arange(len(pdata))
            md_vals = pdata[md_col].to_numpy()
            
            ax.plot(times, md_vals, "o-", color="steelblue", ms=6)
            # OLS trend line
            if len(times) > 1:
                slope = np.polyfit(times, md_vals, 1)
                ax.plot(times, np.polyval(slope, times), "r--", lw=1.5,
                        label=f"{slope[0]:.2f} dB/yr")
            ax.set_title(f"Patient {str(pid)[:8]}", fontsize=9)
            ax.set_xlabel("Visit"); ax.set_ylabel("MD (dB)")
            ax.legend(fontsize=7)
        
        plt.suptitle("Longitudinal MD Trajectories (GRAPE)")
        plt.tight_layout()
        plt.savefig("../reports/eda_grape_trajectories.png", dpi=150, bbox_inches="tight")
        plt.show()

## 5. Clinical Feature Distributions

In [ ]:
if clinical_df is not None:
    # Numeric columns
    num_cols = [c for c in clinical_df.columns if clinical_df[c].dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]]
    print(f"Numeric columns: {num_cols[:10]}")
    print(clinical_df[num_cols[:8]].describe())

    # Plot distributions
    n_cols = min(8, len(num_cols))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, col in zip(axes.flat, num_cols[:n_cols]):
        data = clinical_df[col].drop_nulls().to_numpy()
        ax.hist(data, bins=30, color="teal", edgecolor="white", alpha=0.8)
        ax.set_title(col)
        ax.set_xlabel(col)
    plt.suptitle("GRAPE Clinical Feature Distributions")
    plt.tight_layout()
    plt.savefig("../reports/eda_grape_clinical.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Summary

Key findings to inform our modeling approach:

In [ ]:
print("=== GRAPE EDA Summary ===")
print(f"Data dir: {GRAPE_DIR.resolve()}")
print(f"CSV files: {len(csv_files)}")
print(f"Images:   {len(image_files)}")

print("""
Action items based on EDA:
  [ ] Identify myopic patients (axial_length >= 26mm or SE <= -6D)
  [ ] Map GRAPE column names to our preprocessing schema
  [ ] Check image resolution → resize strategy for ConvNeXt (224×224)
  [ ] Verify progression annotation format and units
  [ ] Plan train/val/test split (stratify by progression rate)

→ Next: 04_vf_preprocessing.ipynb
""")